# OU model workflow


In [ ]:
from pathlib import Path
from numbers import Integral, Real
import hashlib
import importlib
import itertools
import json
import platform
import sys
from tqdm.auto import tqdm

import matplotlib
import matplotlib.pyplot as plt
import scienceplots
plt.style.use('science')
import numpy as np
import pandas as pd
from IPython.display import display

MODEL_DIR = next(
    root / "paper/section_1/OU-comparison"
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "paper/section_1/OU-comparison/OU_model.py").is_file()
)
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
import OU_model as ou
importlib.reload(ou)
print(f"Model: {ou.__file__}\nPython: {sys.executable}")

### Settings

Edit the grid and sampling settings here. `dt` is the observation interval;
`max_lag` counts frames. Use `initial = "zero"` for the ABM startup transient.
Units label values without converting them. Choose another `output` directory
to keep a separate run; this notebook replaces the same five output files.

In [ ]:
replicates = 64
duration = 360
dt = 3
max_lag = 10
seed = 309
initial = "zero"
time_unit, length_unit = "min", "µm"
output = MODEL_DIR / "outputs"

alphas = np.linspace(0.01, 10, 10).tolist()
sigma_mus = np.linspace(0, 300, 9).tolist()
sigma_rs = np.linspace(0, 10, 9).tolist()

print(
    f"Replicates: {replicates}\nDuration: {duration} {time_unit}\nTime step: {dt} {time_unit}\nMax lag: {max_lag}\nSeed: {seed}\nInitial condition: {initial}\nAlpha values: {alphas}\nPolarity noise (sigma_mu) values: {sigma_mus}\nTranslational noise (sigma_r) values: {sigma_rs}"
    )

###
# Pre-check of the parameters to avoid wasting time on invalid configurations. 
# ###

for name, values, positive in (
    ("alphas", alphas, True),
    ("sigma_mus", sigma_mus, False),
    ("sigma_rs", sigma_rs, False)
    ):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or not len(values) or not np.isfinite(values).all() or np.any(values <= 0 if positive else values < 0):
        raise ValueError(f"{name}: use a nonempty list of finite {'positive' if positive else 'non-negative'} numbers")
    if len(np.unique(values)) != len(values):
        raise ValueError(f"{name}: duplicate entries would bias grid weighting")
for name, value in (("duration", duration), ("dt", dt)):
    if isinstance(value, bool) or not isinstance(value, Real) or not np.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive")
if not np.isclose(duration / dt, round(duration / dt), rtol=0, atol=1e-8):
    raise ValueError("duration must be an integer multiple of dt")
n_steps = round(duration / dt)
for name, value, minimum in (("replicates", replicates, 2), ("max_lag", max_lag, 3), ("seed", seed, 0)):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{name} must be an integer >= {minimum}")
if max_lag >= n_steps or initial not in {"stationary", "zero"}:
    raise ValueError("Require max_lag < duration/dt and initial = stationary or zero")
if max(sigma_mus) == max(sigma_rs) == 0:
    raise ValueError("The grid must include nonzero motion")

In [ ]:
preview_parameters = dict(alpha=0.3, sigma_mu=1.0, sigma_r=0.3)
positions = ou.simulate_ou(
    **preview_parameters, n_tracks=10, n_steps=n_steps,
    dt=dt, rng=np.random.default_rng(seed), initial=initial,
)
fig, ax = plt.subplots(figsize=(6, 5))
for trajectory in positions:
    ax.plot(*trajectory.T, lw=1)
ax.scatter(0, 0, color="black", s=15)
ax.set(
    xlabel=f"x ({length_unit})", ylabel=f"y ({length_unit})",
    title=(
        rf"$\alpha={preview_parameters['alpha']:g},\ "
        rf"\sigma_\mu={preview_parameters['sigma_mu']:g},\ "
        rf"\sigma_r={preview_parameters['sigma_r']:g}$"), aspect="equal")
plt.show()

### Plot settings and a trajectory preview

In [ ]:
METRIC_LABELS = {
    "observed_duration": "Observed duration",
    "missing_fraction": "Missing observations",
    "mean_speed": "Mean speed",
    "median_speed": "Median speed",
    "speed_cv": "Speed coefficient of variation",
    "speed_tail_ratio": "Speed-tail ratio (95th / 50th percentile)",
    "mean_turning_angle_deg": "Mean turning angle (degrees)",
    "directional_persistence": "Directional persistence",
    "reversal_fraction": "Reversal fraction",
    "path_length": "Observed path length",
    "net_displacement": "Net displacement",
    "straightness": "Track straightness",
    "radius_gyration": "Radius of gyration",
    "msd_exponent": "MSD exponent",
    "msd_first_lag": "MSD at first lag",
    "msd_last_lag": "MSD at maximum lag",
    "non_gaussian_first_lag": "Non-Gaussian parameter at first lag",
}

ACQUISITION = {"observed_duration", "missing_fraction"}

CURVE_LABELS = {
    "msd": "Time-averaged MSD", "directional_acf": "Directional autocorrelation",
    "velocity_acf": "Velocity autocorrelation", "speed_acf": "Speed autocorrelation",
    "turning_cdf": "Turning-angle cumulative distribution",
}

BLUE, GREY = "#3775BA", "#CFCECE"


def metric_label(key):
    label = METRIC_LABELS[key]
    if key in {"mean_speed", "median_speed"}:
        label += f" ({length_unit}/{time_unit})"
    elif key in {"path_length", "net_displacement", "radius_gyration"}:
        label += f" ({length_unit})"
    elif key in {"msd_first_lag", "msd_last_lag"}:
        label += f" ({length_unit}²)"
    return label

### Check the model and run the grid

This cell calls the numerical functions from `OU_model.py` and keeps the sweep
results in memory. Every moving combination receives the same number of tracks.

In [ ]:
ou.self_check()

parameters = [
    (alpha, sigma_mu, sigma_r)
    for alpha, sigma_mu, sigma_r in itertools.product(alphas, sigma_mus, sigma_rs)
    if sigma_mu > 0 or sigma_r > 0
]
scalar_batches = []
curve_batches = {}

for parameter_id, (alpha, sigma_mu, sigma_r) in enumerate(
    tqdm(parameters, desc="Simulating", unit="combination")
):
    rng = np.random.default_rng(np.random.SeedSequence([seed, parameter_id, 0]))
    paths = ou.simulate_ou(
        alpha, sigma_mu, sigma_r, replicates, n_steps, dt, rng, initial
    )
    scalars, track_curves = ou.summarize_tracks(paths, dt, max_lag)
    scalar_batches.append(pd.DataFrame(dict(
        parameter_id=parameter_id,
        alpha=alpha, sigma_mu=sigma_mu, sigma_r=sigma_r,
        replicate=np.arange(replicates),
        **scalars,
    )))

    for name, values in track_curves.items():
        curve_batches.setdefault(name, []).append(values.astype(np.float32))

statistics = pd.concat(scalar_batches, ignore_index=True)
curves = {name: np.concatenate(batches) for name, batches in curve_batches.items()}
del scalar_batches, curve_batches

## 4. Summary statistics and plots

Scalar and curve rows refer to the same tracks. Parameter IDs retain their joint
information. Bands summarize the finite grid; they are not confidence intervals.

In [ ]:
curve_summary = pd.DataFrame(ou.summarize_curves(curves, dt, max_lag))
metrics = [name for name in METRIC_LABELS if name not in ACQUISITION]
print(f"{len(statistics):,} tracks; {len(parameters)} parameter combinations")
display(statistics.head())
display(statistics[metrics].describe(percentiles=[0.05, 0.5, 0.95]).T)
display(curve_summary.head())

In [ ]:
keys = [key for key in METRIC_LABELS if key not in ACQUISITION]
fig, axes = plt.subplots(5, 3, figsize=(13, 16), layout="constrained")
logarithmic = {"mean_speed", "median_speed", "path_length", "net_displacement",
               "radius_gyration", "msd_first_lag", "msd_last_lag"}
for ax, key in zip(axes.flat, keys):
    values = statistics[key].dropna().to_numpy()
    if len(values):
        # Limit plotted points while retaining the sampled extrema.
        probability = np.linspace(0, 1, min(2001, len(values)))
        ax.plot(np.quantile(values, probability), probability, color=BLUE, lw=1.8)
        if key in logarithmic and values.min() > 0:
            ax.set_xscale("log")
    else:
        ax.text(.5, .5, "Undefined for this grid", ha="center", transform=ax.transAxes)
    if initial == "stationary" and key in {"speed_cv", "speed_tail_ratio", "non_gaussian_first_lag"}:
        theoretical = {"speed_cv": np.sqrt((4 - np.pi) / np.pi),
                       "speed_tail_ratio": np.sqrt(np.log(20) / np.log(2)),
                       "non_gaussian_first_lag": 0}[key]
        ax.axvline(theoretical, color="#767676", ls="--", lw=1)
    ax.set(xlabel=metric_label(key), ylabel="Cumulative fraction of tracks", ylim=(0, 1))
subtitle = "Equal weight per grid point"
if initial == "stationary":
    subtitle += "; dashed lines: stationary population values"
fig.suptitle("Pure OU motion: finite-track statistic distributions\n" + subtitle, fontsize=14)
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / "scalar_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8), layout="constrained")
for ax, (name, label) in zip(axes.flat, CURVE_LABELS.items()):
    table = curve_summary[curve_summary.curve == name]
    x = table.lag_or_angle.to_numpy()
    ax.fill_between(x, table["min"], table["max"], color=GREY, alpha=.35, label="Sampled min–max")
    ax.fill_between(x, table.q05, table.q95, color=BLUE, alpha=.20, label="Track 5–95%")
    ax.plot(x, table["median"], color=BLUE, lw=1.8, label="Track median")
    ax.set_title(label)
    ax.set_xlabel("Unsigned turning angle (degrees)" if name == "turning_cdf" else f"Lag ({time_unit})")
    if name == "msd":
        ax.set(xscale="log", yscale="log", ylabel=f"MSD ({length_unit}²)")
        ticks = dt * np.unique(np.rint(np.geomspace(1, max_lag, 4)).astype(int))
        ax.set_xticks(ticks, [f"{value:g}" for value in ticks])
        ax.xaxis.set_minor_locator(matplotlib.ticker.NullLocator())
    elif name == "turning_cdf":
        ax.set(xlim=(0, 180), ylim=(0, 1), ylabel="Cumulative fraction")
    else:
        ax.axhline(0, color=".5", lw=.7)
        ax.set_ylabel("Correlation")
legend_ax = axes.flat[-1]
legend_ax.axis("off")
legend_ax.legend(*axes.flat[0].get_legend_handles_labels(), loc="upper left")
legend_ax.text(0, .62, f"{len(statistics):,} simulated tracks\n"
               f"Frame interval: {dt:g} {time_unit}\n"
               f"Track duration: {duration:g} {time_unit}\n\n"
               "Bands describe finite-track variation\nacross the chosen parameter grid.\n"
               "They are not confidence intervals\nor fitted-data predictions.", va="top")
fig.suptitle("Pure OU motion: distribution of trajectory curves", fontsize=14)
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / "curve_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Save numerical results and run settings

In [ ]:
output.mkdir(parents=True, exist_ok=True)
statistics.to_csv(output / "statistics.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})
np.savez_compressed(
    output / "curves.npz", **curves,
    parameter_id=statistics.parameter_id.to_numpy(), replicate=statistics.replicate.to_numpy(),
    lag_times=dt * np.arange(1, max_lag + 1), angle_degrees=ou.TURNING_ANGLE_GRID,
)
settings = dict(
    alphas=alphas, sigma_mus=sigma_mus, sigma_rs=sigma_rs,
    replicates=replicates, duration=duration, dt=dt, max_lag=max_lag, seed=seed,
    initial=initial, time_unit=time_unit, length_unit=length_unit, output=str(output.resolve()),
)
metadata = dict(
    arguments=settings, speed_multiplier=1, n_combinations=len(parameters),
    n_tracks=len(statistics), n_frames=n_steps + 1,
    equations="dmu=-alpha*mu*dt+sigma_mu*dW_mu; dx=mu*dt+sigma_r*dW_r; v=1; independent standard Wiener processes in two Cartesian coordinates",
    method="Exact Gaussian joint transition for polarity and integrated displacement",
    weighting="Equal number of independent trajectories per nonzero-motion grid point",
    scope="Finite grid and finite realizations; no empirical data or claim of empirical agreement",
    metric_definitions=ou.METRIC_DEFINITIONS,
    versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, matplotlib=matplotlib.__version__),
    source_sha256={name: hashlib.sha256((MODEL_DIR / name).read_bytes()).hexdigest()
                   for name in ("OU_model.py", "model_workflow.ipynb")},
    entrypoint="model_workflow.ipynb",
)
(output / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(f"Saved two figures, track statistics, curves and metadata to {output.resolve()}")

Metric definitions are available in `ou.METRIC_DEFINITIONS` and the saved metadata.
The grid is finite and uncalibrated. Agreement with observed trajectories has
not been evaluated.